# 01 — Chat Completions (Layer 1, Part A)

**Goal.** Use the classic **Chat Completions** surface on `gpt-5.4` to answer one adviser question about Avery Chen's portfolio. This is the lowest-level building block in the stack — *no agents, no tools, no orchestration* — and the baseline every subsequent notebook will improve on.

**What you'll see here**

1. A single-turn call with a portfolio snapshot injected into the system prompt.
2. A short multi-turn exchange (manual message-list management).
3. Token streaming to the notebook.
4. A controlled JSON response using `response_format`.

**What's missing on purpose**

- No tool calls — you'd have to wire functions, schemas, and the tool-call loop yourself.
- No server-side conversation state — *you* manage the message list.
- No retrieval — portfolio context is stuffed into the prompt.

Notebook #2 (Responses API) fixes #1 and #2; notebook #3 (Foundry Agent Service) fixes #3.

> Heads-up: Microsoft is steering new work to the **Responses API**. Chat Completions still works for everything in this notebook, but treat it as the legacy baseline.

## Step 1 — Load config, credential, client, and scenario

Same plumbing as notebook #0: `Config` from `.env`, `DefaultAzureCredential`, and an `OpenAI` client pointed at `https://r2d2-foundry-001.openai.azure.com/openai/v1/` with a bearer-token auth flow.

In [ ]:
import sys, pathlib
if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))

from _common.env import load_env
from _common.clients import get_credential, get_openai_client
from _common.scenario import CLIENT_PROFILE, portfolio_summary

cfg = load_env()
credential = get_credential()
client = get_openai_client(cfg, credential=credential)

print("Deployment:", cfg.model_deployment, "| reasoning_effort:", cfg.reasoning_effort)

## Step 2 — Build the adviser system prompt

Chat Completions has no memory, so anything the model needs to know has to be in the message list. We pack a compact JSON-ish snapshot of Avery's profile and current portfolio into the system prompt — fine at this scale, but obviously not how you'd handle a 10,000-client book (that's what retrieval and tools are for in later notebooks).

In [ ]:
import json
from dataclasses import asdict

client_ctx = {
    "profile": asdict(CLIENT_PROFILE),
    "portfolio_summary": portfolio_summary(),
}

SYSTEM_PROMPT = (
    "You are the Cobalt Advisory Co-Pilot, an assistant for licensed wealth advisers at Cobalt Wealth.\n"
    "Be concise, factual, and never give individualized investment advice — explain trade-offs and surface\n"
    "options the adviser can discuss with the client. Always cite figures back to the client_context JSON below.\n"
    "If a question requires data you don't have, say so explicitly.\n\n"
    f"client_context = {json.dumps(client_ctx, default=str, indent=2)}"
)

print(SYSTEM_PROMPT[:600], "\n...[truncated]")

## Step 3 — Single-turn question

Ask one adviser-style question and print the reply plus token usage. `reasoning_effort="low"` keeps latency snappy for short interactive questions; bump it to `"medium"` or `"high"` for harder analysis (notebook #2 will demonstrate this).

In [ ]:
question = (
    "Avery is asking why we still hold so much NVDA. Give me a 4-bullet talking-points list "
    "covering: current concentration vs. her 10% single-stock cap, unrealized gain implications, "
    "ESG/risk fit, and one diversification option to propose."
)

resp = client.chat.completions.create(
    model=cfg.model_deployment,
    reasoning_effort=cfg.reasoning_effort,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ],
)

print("--- Reply ---\n")
print(resp.choices[0].message.content)
print("\n--- Usage ---")
print("prompt:", resp.usage.prompt_tokens, "| completion:", resp.usage.completion_tokens, "| total:", resp.usage.total_tokens)

## Step 4 — Multi-turn (managed by *you*)

Chat Completions is stateless. To have a conversation, you append the assistant's reply to the message list and send the whole list back. Notebook #2 will show how the Responses API replaces this with a `previous_response_id` pointer.

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": "What is Avery's current cash position and how does it compare to her stated 6-month emergency reserve goal?"},
]

def ask(user_text: str) -> str:
    messages.append({"role": "user", "content": user_text})
    r = client.chat.completions.create(
        model=cfg.model_deployment,
        reasoning_effort=cfg.reasoning_effort,
        messages=messages,
    )
    reply = r.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return reply

# Seed turn: the user prompt is already in `messages` above, so call the model once first.
first = client.chat.completions.create(
    model=cfg.model_deployment,
    reasoning_effort=cfg.reasoning_effort,
    messages=messages,
)
messages.append({"role": "assistant", "content": first.choices[0].message.content})
print("Turn 1:\n", first.choices[0].message.content)

print("\nTurn 2:\n", ask("Given that, what's a reasonable upper bound on how much of CASH I could redeploy this quarter without breaking her constraints?"))
print("\nTurn 3:\n", ask("Summarize this conversation as three bullet points for my CRM note."))

## Step 5 — Streaming

For chat-style UIs you want tokens as they're produced. Set `stream=True` and iterate the chunks. The shape is `chunk.choices[0].delta.content` — concatenate, render, done.

In [ ]:
stream = client.chat.completions.create(
    model=cfg.model_deployment,
    reasoning_effort=cfg.reasoning_effort,
    stream=True,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "Draft a 3-sentence client-friendly email to Avery confirming her next portfolio review meeting and the two topics we'll cover: NVDA concentration and cash redeployment."},
    ],
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

## Step 6 — Structured (JSON) output

When downstream code needs to *consume* the reply (not display it), force JSON with `response_format={"type": "json_object"}` and ask the model to fill a shape you specify in the prompt. For richer schema enforcement, notebook #2 will use `response_format={"type":"json_schema", ...}`.

In [ ]:
schema_hint = {
    "summary": "one-sentence headline",
    "concentration_flags": [
        {"symbol": "...", "pct_of_portfolio": 0.0, "breaches_10pct_cap": False}
    ],
    "recommended_next_actions": ["..."],
}

resp = client.chat.completions.create(
    model=cfg.model_deployment,
    reasoning_effort=cfg.reasoning_effort,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            "Return a JSON object that conforms to this shape (keys and types must match exactly): "
            f"{json.dumps(schema_hint)}. Populate it from Avery's current portfolio."
        )},
    ],
)

parsed = json.loads(resp.choices[0].message.content)
print(json.dumps(parsed, indent=2))

## Recap & what's next

You've now exercised every meaningful Chat Completions feature against `gpt-5.4`: single-turn, multi-turn with manual state, streaming, and JSON-shaped output. Notice the friction points:

- **You** carry the conversation history on every call.
- **You** stuffed portfolio context into the system prompt.
- There's no way for the model to *call back* into your code (tools), do web search, or read a PDF — without you writing the loop yourself.

Next: **`02_responses_api.ipynb`** — same questions, but on the **Responses API** with server-side state (`previous_response_id`), built-in `web_search`, and PDF input. This is the surface Microsoft is steering new work toward.